# 20210808053 Data Mining Final Project

This notebook summarizes the clean submission version of the Computer Science Journal Finder project. It uses the project package under `src/` and the saved artifacts under `exports/20210808053/`.

## Submission Mapping

| Requirement | File / module |
|---|---|
| Source code | `src/`, `app.py` |
| Jupyter Notebook | `notebooks/20210808053_Final_Project.ipynb` |
| IEEE report | `report/20210808053_IEEE_PROJECT_REPORT.tex` |
| Top-5 journal recommender | `src/final_project/recommender_model.py`, saved pipeline |
| Topic clustering | `src/final_project/topic_modeling.py`, saved cluster CSV |
| Software interface | `python -m streamlit run app.py` |

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_dataset, summarize_dataset
from src.final_project.clustering import load_clustered_data, summarize_cluster
from src.final_project.paths import DB_PATH, PIPELINE_PATH, RECOMMENDER_META_PATH, CLUSTERED_DATASET_PATH
from src.final_project.recommender_model import load_pipeline, recommend_journals

ROOT = Path.cwd()
print('Project root:', ROOT)
print('SQLite:', DB_PATH)
print('Saved model:', PIPELINE_PATH)
print('Cluster CSV:', CLUSTERED_DATASET_PATH)

## Dataset Summary

The project uses the converted SQLite database `CompSciencePub.sqlite`. The loader joins article, abstract, journal, keyword, keyword plus, and subject fields, then builds normalized training text.

In [ ]:
frame = load_dataset(DB_PATH)
summary = summarize_dataset(frame)
pd.Series(summary, name='count')

## Saved Final Recommender Metrics

The final submitted app loads the saved multi-channel TF-IDF + `SGDClassifier(loss='log_loss')` pipeline. The metadata file stores the holdout evaluation results.

In [ ]:
meta = json.loads(RECOMMENDER_META_PATH.read_text(encoding='utf-8-sig'))
display(pd.Series(meta, name='value'))

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Top-1', 'Top-5'], [meta['holdout_top1_accuracy'], meta['holdout_top5_accuracy']], color=['#4c72b0', '#55a868'])
ax.set_ylim(0, 1)
ax.set_ylabel('Holdout accuracy')
ax.set_title('Final recommender performance')
for idx, value in enumerate([meta['holdout_top1_accuracy'], meta['holdout_top5_accuracy']]):
    ax.text(idx, value + 0.02, f'{value:.3f}', ha='center')
plt.tight_layout()
plt.show()

## Top-5 Journal Recommendation Demo

This demo uses the same saved model loaded by the Streamlit app.

In [ ]:
pipeline = load_pipeline()
example = """
This paper proposes a graph neural network model for mining software repository data and predicting source code defects. The experiments evaluate precision, recall, and generalization on open source projects.
"""
recommend_journals(
    pipeline,
    abstract=example,
    title='Graph neural networks for software defect prediction',
    keywords='software mining, defect prediction, graph neural networks',
    subjects='Artificial Intelligence, Software Engineering',
    top_k=5,
)

## Topic Cluster Summary

The dashboard uses the saved `step9_clustered_dataset.csv` artifact. Each record has a KMeans cluster label.

In [ ]:
clusters = load_clustered_data()
print('Clustered rows:', len(clusters))
print('Number of clusters:', clusters['cluster'].nunique())
clusters['cluster'].value_counts().sort_index().rename('article_count').to_frame()

In [ ]:
summary = summarize_cluster(clusters, int(clusters['cluster'].mode().iloc[0]))
print('Cluster name:', summary['name'])
print('Article count:', summary['article_count'])
print('Keywords:', ', '.join(summary['keywords']))
display(summary['top_journals'].head(10))
display(summary['sample_articles'].head(10))

## Rebuild Commands

The clean submission keeps only the final saved model and cluster output. Intermediate CSV files can be regenerated when needed:

```powershell
python -m src.final_project.enrichment
python -m src.final_project.training
python -m src.final_project.topic_modeling
```